# 05 — Train OnsetsAndFrames (jongwook architecture)

This notebook trains the full OnsetsAndFrames model (jongwook variant with all
5 improvements over the 2018a paper).

**Training stages in this notebook:**

| Stage | `max_files` | Approx. data | Purpose |
|-------|-------------|--------------|----------|
| 0 — smoke | 5 | ~10 min | Verify training loop runs without errors |
| 1 — tiny | 30 | ~1 hr | Check model learns at all |
| 2 — small | 100 | ~4 hr | Tune hyperparameters |
| 3 — medium | 400 | ~20 hr | Compare O&F vs O&F-adv |
| 4 — full | None (all) | ~178 hr | Final results for dissertation |

Run stages 0→3 in order. Only run stage 4 after stage 3 confirms the model
scales correctly.

**All checkpoints are saved to Google Drive** so they survive Colab restarts.
Every run auto-creates a `runs/<run_name>/` directory with:
- `checkpoints/` — all saved `.pt` files + `best.pt`
- `plots/` — loss curve PNG after every epoch
- `metrics.json` — all epoch losses, updated live
- `config.json` — exact training config used

## Cell 1 — Environment setup

Run every session. Mounts Drive, installs packages, clones repo, sets paths.

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU! Change runtime to T4 GPU."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")

In [ ]:
# ── Install packages ───────────────────────────────────────────────────────
!pip install -q torch torchaudio pretty_midi pandas tqdm matplotlib pyyaml mir_eval
print("Packages installed ✓")

In [ ]:
# ── Clone / update repo ────────────────────────────────────────────────────
import os
from getpass import getpass

REPO_DIR = '/content/AMT_FYP'
TOKEN    = getpass('GitHub token: ')

if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://{TOKEN}@github.com/Mobinmo83/AMT_FYP.git {REPO_DIR}")
    print(f"Cloned → {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")
    print(f"Pulled latest → {REPO_DIR}")

In [ ]:
# ── sys.path + project root ────────────────────────────────────────────────
import sys

PROJECT_ROOT = '/content/AMT_FYP/piano_amt'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Quick sanity check
from src.constants import N_MELS, FRAMES_PER_SECOND, N_KEYS
assert N_MELS == 229 and abs(FRAMES_PER_SECOND - 31.25) < 1e-6 and N_KEYS == 88
print("Imports OK ✓")

In [ ]:
# ── Path constants  (edit ONLY these if your Drive layout differs) ──────────
DRIVE_ROOT   = '/content/drive/MyDrive/piano_amt'
MAESTRO_ROOT = f'{DRIVE_ROOT}/maestro-v3.0.0'
CACHE_DIR    = f'{DRIVE_ROOT}/cache'
RUNS_DIR     = f'{DRIVE_ROOT}/runs'

import os
os.makedirs(RUNS_DIR, exist_ok=True)

# Verify cache exists
import glob
npzs = glob.glob(f'{CACHE_DIR}/*.npz')
print(f"Cache files : {len(npzs)}")
assert len(npzs) > 0, "Run 02_build_cache.ipynb first!"
print(f"MAESTRO root: {MAESTRO_ROOT}")
print(f"Runs dir    : {RUNS_DIR}")

## Cell 2 — Training configuration

**Edit the `CONFIG` dict to choose your training stage.**

Key parameters:
- `run_name`: unique name saved to Drive — use descriptive names like `of_baseline_stage1`
- `max_files`: controls dataset size (None = full 962 training files)
- `model_complexity`: 48 = full jongwook model (~26M params); 16 = small debug model (~1M params)
- `epochs`: use fewer epochs for smoke/tiny tests; 30+ for real training
- `resume`: path to a `.pt` checkpoint to resume from; None to start fresh

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TRAINING CONFIGURATION — edit this cell
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── Identity ───────────────────────────────────────────────────────────
    # Choose a descriptive name; results saved to RUNS_DIR/<run_name>/
    'run_name':   'of_baseline_smoke',   # <-- CHANGE THIS each run

    # ── Dataset size ──────────────────────────────────────────────────────
    # Stage 0 (smoke): max_files=5     (~10 min audio, 2 epochs, verify loop)
    # Stage 1 (tiny):  max_files=30    (~1 hr audio, 5 epochs)
    # Stage 2 (small): max_files=100   (~4 hr audio, 15 epochs)
    # Stage 3 (medium):max_files=400   (~20 hr audio, 30 epochs)
    # Stage 4 (full):  max_files=None  (~178 hr audio, 30+ epochs)
    'max_files':  5,

    # ── Model ─────────────────────────────────────────────────────────────
    # 48 = full jongwook (~26M params) — use for stage 2+
    # 16 = small debug (~1M params) — use for stage 0+1 to move faster
    'model_complexity': 16,

    # ── Training ─────────────────────────────────────────────────────────
    'epochs':        2,      # smoke=2, tiny=5, small=15, medium=30, full=30+
    'batch_size':    4,      # 4 for smoke/tiny, 8 for small/medium/full
    'lr':            6e-4,   # Hawthorne 2018a §3.2
    'pos_weight':    5.0,
    'max_grad_norm': 3.0,
    'num_workers':   2,
    'log_every':     10,     # print loss every N steps

    # ── Resume from checkpoint (set to None to start fresh) ───────────────
    # Example: f'{RUNS_DIR}/of_baseline_stage1/checkpoints/best.pt'
    'resume': None,
}

# ─── Print config summary ─────────────────────────────────────────────────
print("Training configuration:")
for k, v in CONFIG.items():
    print(f"  {k:20s}: {v}")

## Cell 3 — Build model and inspect

In [ ]:
import torch
from models.onsets_frames.model import OnsetsAndFrames

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = OnsetsAndFrames(model_complexity=CONFIG['model_complexity'])

n_params = model.count_parameters()
print(f"Model          : OnsetsAndFrames")
print(f"model_complexity: {CONFIG['model_complexity']}")
print(f"Parameters     : {n_params:,}")
print(f"Device         : {device}")

# Quick forward pass to check shapes
model = model.to(device)
dummy = torch.zeros(2, 229, 640).to(device)
with torch.no_grad():
    out = model(dummy)
for k, v in out.items():
    print(f"  {k:10s}: {tuple(v.shape)}  min={v.min():.3f} max={v.max():.3f}")

assert out['onset'].shape  == (2, 640, 88), "Onset shape wrong"
assert out['frame'].shape  == (2, 640, 88), "Frame shape wrong"
assert out['offset'].shape == (2, 640, 88), "Offset shape wrong"
print("\nModel shapes OK ✓")

## Cell 4 — Build DataLoaders

In [ ]:
from src.dataloader import get_dataloader

train_loader = get_dataloader(
    maestro_root   = MAESTRO_ROOT,
    split          = 'train',
    batch_size     = CONFIG['batch_size'],
    num_workers    = CONFIG['num_workers'],
    cache_dir      = CACHE_DIR,
    max_files      = CONFIG['max_files'],
    use_augmentation = True,
    pin_memory     = True,
)
val_loader = get_dataloader(
    maestro_root   = MAESTRO_ROOT,
    split          = 'validation',
    batch_size     = CONFIG['batch_size'],
    num_workers    = CONFIG['num_workers'],
    cache_dir      = CACHE_DIR,
    max_files      = CONFIG['max_files'],
    use_augmentation = False,
    pin_memory     = True,
)
print(f"Train batches : {len(train_loader)}  (batch_size={CONFIG['batch_size']})")
print(f"Val batches   : {len(val_loader)}")

# Peek at one batch
batch = next(iter(train_loader))
print(f"\nSample batch:")
for k, v in batch.items():
    if hasattr(v, 'shape'):
        print(f"  {k:12s}: {tuple(v.shape)}")
print("DataLoaders OK ✓")

## Cell 5 — Train

This cell runs the full training loop.  
Loss curves are saved to Drive after every epoch — if Colab disconnects,
you can resume from `best.pt` by setting `CONFIG['resume']` above.

**Watch:** the `[ep NNN step NNN]` lines show per-batch loss in real time.

In [ ]:
from models.onsets_frames.train import OnsetsFramesLoss, Trainer, RunDirectory

# Create run directory
run_dir = RunDirectory(RUNS_DIR, CONFIG['run_name'])

# Save config
import json
cfg_to_save = dict(CONFIG)
cfg_to_save.update({'maestro_root': MAESTRO_ROOT, 'cache_dir': CACHE_DIR,
                    'device': str(device)})
run_dir.save_config(cfg_to_save)

# Fresh model (or load from resume)
model = OnsetsAndFrames(model_complexity=CONFIG['model_complexity']).to(device)

start_epoch = 1
if CONFIG['resume']:
    ckpt = torch.load(CONFIG['resume'], map_location=device)
    model.load_state_dict(ckpt['model_state'])
    start_epoch = ckpt['epoch'] + 1
    print(f"Resumed from epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

trainer = Trainer(
    model          = model,
    train_loader   = train_loader,
    val_loader     = val_loader,
    device         = device,
    run_dir        = run_dir,
    lr             = CONFIG['lr'],
    pos_weight     = CONFIG['pos_weight'],
    max_grad_norm  = CONFIG['max_grad_norm'],
    log_every      = CONFIG['log_every'],
)

trainer.fit(epochs=CONFIG['epochs'])

print(f"\n✓ Training complete. Run directory: {run_dir.root}")
print(f"  Best val loss : {trainer.best_val_loss:.4f}")
print(f"  Best checkpoint: {run_dir.best_checkpoint_path()}")

## Cell 6 — Visualise training curves

Plot the saved loss curves from this run. Also saves a final high-res PNG
for the report.

In [ ]:
import matplotlib
matplotlib.use('Agg')   # headless
import matplotlib.pyplot as plt
import json

from evaluate.plots import plot_training_curves

metrics_path = run_dir.root / 'metrics.json'
save_path    = run_dir.plots / 'training_curves_final.png'

plot_training_curves(
    metrics_path = metrics_path,
    save_path    = save_path,
    title        = f"Training curves — {CONFIG['run_name']}",
)

# Also display in notebook
from IPython.display import Image
Image(str(save_path))

## Cell 7 — Evaluate on validation set

Runs the best checkpoint on the validation split and computes all AMT metrics.
Results saved to `runs/<run_name>/eval_validation/`.

In [ ]:
from models.onsets_frames.evaluate import run_evaluation

best_ckpt = str(run_dir.best_checkpoint_path())
print(f"Evaluating: {best_ckpt}")

val_summary = run_evaluation(
    checkpoint_path  = best_ckpt,
    maestro_root     = MAESTRO_ROOT,
    cache_dir        = CACHE_DIR,
    split            = 'validation',
    max_files        = CONFIG['max_files'],    # same subset for fair comparison
    save_midi        = True,
    save_plots       = True,
    model_complexity = CONFIG['model_complexity'],
)

print("\nValidation summary:")
for k in ['onset_f1', 'frame_f1', 'note_with_offset_f1', 'note_with_offset_vel_f1',
          'ea_offset_mae_ms', 'ea_chord_completeness']:
    print(f"  {k:35s}: {val_summary.get(k, 0):.4f}")

## Cell 8 — Evaluate on test set

Only run this once per stage (after hyperparameters are finalised).
Results saved to `runs/<run_name>/eval_test/`.

In [ ]:
test_summary = run_evaluation(
    checkpoint_path  = best_ckpt,
    maestro_root     = MAESTRO_ROOT,
    cache_dir        = CACHE_DIR,
    split            = 'test',
    max_files        = CONFIG['max_files'],
    save_midi        = True,
    save_plots       = True,
    model_complexity = CONFIG['model_complexity'],
)

print("\nTest summary:")
for k in ['onset_f1', 'frame_f1', 'note_with_offset_f1',
          'ea_offset_mae_ms', 'ea_chord_completeness']:
    print(f"  {k:35s}: {test_summary.get(k, 0):.4f}")

## Cell 9 — Compare all runs

After training multiple stages / variants, use this cell to generate a
comparison table across all runs. Outputs CSV, LaTeX table, and bar chart.

In [ ]:
from evaluate.compare import (
    compare_all_runs, print_comparison_table,
    save_comparison_csv, latex_table, plot_comparison_bar
)
import os

COMPARISON_DIR = f'{DRIVE_ROOT}/comparison'
os.makedirs(COMPARISON_DIR, exist_ok=True)

# Load all test results
rows = compare_all_runs(RUNS_DIR, split='test')

if rows:
    print_comparison_table(rows)

    save_comparison_csv(rows, f'{COMPARISON_DIR}/comparison_test.csv')

    print("\n--- LaTeX table ---")
    print(latex_table(rows))

    plot_comparison_bar(
        rows,
        save_path = f'{COMPARISON_DIR}/comparison_test.png',
        title     = 'Model comparison — test set'
    )

    from IPython.display import Image
    Image(f'{COMPARISON_DIR}/comparison_test.png')
else:
    print("No test results found yet. Run evaluation cells first.")

## Cell 10 — Quick MIDI transcription demo

Load any audio file, run the best checkpoint, and save the decoded MIDI.
Useful for showing the system works end-to-end in your dissertation demo.

In [ ]:
import glob, torch
from src.audio import load_audio_as_log_mel
from src.dataloader import sliding_windows
from src.constants import FRAMES_PER_SECOND, N_KEYS, MAX_SEGMENT_FRAMES
from models.onsets_frames.model import OnsetsAndFrames
from models.onsets_frames.decode import rolls_to_midi_file

# ── Choose audio file ───────────────────────────────────────────────────────
# Use any .wav from MAESTRO or your own recording
test_wavs = glob.glob(f'{MAESTRO_ROOT}/**/*.wav', recursive=True)
AUDIO_PATH = test_wavs[0]   # first available file
OUT_MIDI   = f'/content/transcription_demo.mid'
print(f"Transcribing: {AUDIO_PATH}")

# ── Load model ──────────────────────────────────────────────────────────────
best_ckpt_path = str(run_dir.best_checkpoint_path())
model = OnsetsAndFrames(model_complexity=CONFIG['model_complexity'])
ckpt  = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model = model.to(device).eval()

# ── Compute mel ─────────────────────────────────────────────────────────────
log_mel = load_audio_as_log_mel(AUDIO_PATH, device=device)  # (229, T)
T_full  = log_mel.shape[1]
print(f"Audio length: {T_full / FRAMES_PER_SECOND:.1f} s  ({T_full} frames)")

# ── Sliding window inference ────────────────────────────────────────────────
pred_onset    = torch.zeros(T_full, N_KEYS)
pred_frame    = torch.zeros(T_full, N_KEYS)
pred_velocity = torch.zeros(T_full, N_KEYS)

windows = sliding_windows(log_mel, window_frames=MAX_SEGMENT_FRAMES)
with torch.no_grad():
    for win in windows:
        w_mel  = win['mel'].unsqueeze(0).to(device)    # (1,229,640)
        start  = win['start_frame']
        end    = min(start + MAX_SEGMENT_FRAMES, T_full)
        out    = model(w_mel)
        pred_onset[start:end]    = out['onset'][0, :end-start].cpu()
        pred_frame[start:end]    = out['frame'][0, :end-start].cpu()
        pred_velocity[start:end] = out['velocity'][0, :end-start].cpu()

print(f"Inference done. Decoding to MIDI...")

# ── Decode + save ───────────────────────────────────────────────────────────
rolls_to_midi_file(
    onset_roll    = pred_onset,
    frame_roll    = pred_frame,
    velocity_roll = pred_velocity,
    output_path   = OUT_MIDI,
    fps           = FRAMES_PER_SECOND,
)
print(f"MIDI saved to: {OUT_MIDI}")
print("Download it from the Colab file browser (left panel) or copy to Drive.")

## Cell 11 — Copy run to Drive (backup)

Run this if your RUNS_DIR is on local Colab disk rather than Drive directly.
Skip if RUNS_DIR is already on Drive (which is the default above).

In [ ]:
# Only needed if RUNS_DIR was set to a local path like /content/runs
# By default RUNS_DIR = DRIVE_ROOT/runs, so this cell is usually a no-op.

import shutil, os
LOCAL_RUNS = '/content/runs'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'

if os.path.exists(LOCAL_RUNS) and LOCAL_RUNS != DRIVE_RUNS:
    print(f"Copying {LOCAL_RUNS} → {DRIVE_RUNS}")
    shutil.copytree(LOCAL_RUNS, DRIVE_RUNS, dirs_exist_ok=True)
    print("Backup complete ✓")
else:
    print("RUNS_DIR is already on Drive — no copy needed ✓")

---

## Scaling guide — what to change at each stage

```
Stage 0 — smoke (verify loop)
  max_files=5, model_complexity=16, epochs=2, batch_size=4
  Expected: training runs without error, val_loss finite

Stage 1 — tiny (model learns)
  max_files=30, model_complexity=16, epochs=5, batch_size=4
  Expected: val_loss decreases over epochs

Stage 2 — small (hyperparameter check)
  max_files=100, model_complexity=48, epochs=15, batch_size=8
  Expected: onset_f1 > 0.5 on validation

Stage 3 — medium (dissertation comparison)
  max_files=400, model_complexity=48, epochs=30, batch_size=8
  Expected: onset_f1 ≈ 0.8, frame_f1 ≈ 0.7  (comparable to paper)

Stage 4 — full (final results)
  max_files=None, model_complexity=48, epochs=30+, batch_size=8
  Expected: results matching Hawthorne 2018a / jongwook reported values
```

**Resume pattern** (if Colab disconnects mid-training):
```python
CONFIG['resume'] = f'{RUNS_DIR}/of_baseline_stage3/checkpoints/best.pt'
CONFIG['epochs'] = 30   # will train for 30 more epochs from that checkpoint
```